# 32. Reinforcement Learning: Policy Gradient Methods

## Algorithm Category
**Type**: Reinforcement Learning - Policy-Based  
**Complexity**: High  
**Use Case**: Direct policy optimization using gradient ascent on expected return

## Learning Objectives

By the end of this notebook, you will be able to:
- Understand policy gradient methods and REINFORCE algorithm
- Implement REINFORCE from scratch
- Understand policy gradient theorem
- Compare policy-based vs value-based methods
- Visualize policy learning and reward improvement
- Apply policy gradients to simple environments

## Historical Context

Policy gradient methods were developed in the 1990s-2000s:
- Williams, R.J. (1992): "Simple statistical gradient-following algorithms"
- REINFORCE algorithm (1992)
- Foundation for modern policy gradient methods (A3C, PPO, etc.)

**Key Papers/References:**
- Williams, R.J. (1992). "Simple statistical gradient-following algorithms for connectionist reinforcement learning"
- Sutton, R.S., et al. (2000). "Policy gradient methods for reinforcement learning"

## When to Use Policy Gradient Methods

Policy gradients are appropriate when:
- You have continuous action spaces
- You need stochastic policies
- Working with high-dimensional state spaces
- Value function is hard to estimate
- You want direct policy optimization
- Good for complex control problems

## Theory & Mechanics

### Mathematical Foundation

Policy gradient methods optimize the policy directly using gradient ascent.

**Policy Gradient Theorem:**
$$\nabla_\theta J(\theta) = \mathbb{E}_{\pi_\theta}[\nabla_\theta \log \pi_\theta(a|s) Q^{\pi_\theta}(s, a)]$$

**REINFORCE Algorithm:**
$$\nabla_\theta J(\theta) = \mathbb{E}[\nabla_\theta \log \pi_\theta(a|s) G_t]$$

Where $G_t$ is the return (sum of rewards from time t).

**Policy Update:**
$$\theta \leftarrow \theta + \alpha \nabla_\theta J(\theta)$$

**REINFORCE Update (Monte Carlo):**
$$\theta \leftarrow \theta + \alpha \gamma^t G_t \nabla_\theta \log \pi_\theta(a_t|s_t)$$

### How It Works

1. **Initialize**: Random policy parameters $\theta$
2. **Generate episode**: Follow current policy to collect trajectory
3. **Calculate returns**: Compute discounted returns $G_t$ for each step
4. **Update policy**: Gradient ascent on expected return
5. **Repeat**: Steps 2-4 until convergence

### Key Hyperparameters

- **learning_rate**: Step size for policy updates
- **gamma**: Discount factor for future rewards
- **baseline**: Variance reduction technique (optional)

### Advantages

- Works with continuous action spaces
- Can learn stochastic policies
- Direct policy optimization
- No need for value function
- Better for high-dimensional spaces

### Limitations

- High variance in gradient estimates
- Slow convergence
- Requires many samples
- May converge to local optima
- Sample inefficient


## Implementation

Let's implement REINFORCE algorithm from scratch.


In [ ]:
# Import necessary libraries
import numpy as np
import matplotlib.pyplot as plt
from collections import deque

print("Libraries imported successfully!")


In [ ]:
# Simple CartPole-like environment
class SimpleEnv:
    def __init__(self):
        self.state = 0  # Position
        self.max_steps = 20
        
    def reset(self):
        self.state = 0
        self.steps = 0
        return self.state
    
    def step(self, action):
        """Action: 0=left, 1=right"""
        if action == 0:
            self.state -= 1
        else:
            self.state += 1
        
        self.steps += 1
        
        # Reward: closer to center (0) is better
        reward = -abs(self.state)
        
        # Done if reached center or max steps
        done = (self.state == 0) or (self.steps >= self.max_steps)
        
        return self.state, reward, done

# REINFORCE Agent
class REINFORCEAgent:
    def __init__(self, n_states, n_actions, learning_rate=0.01, gamma=0.99):
        self.n_states = n_states
        self.n_actions = n_actions
        self.lr = learning_rate
        self.gamma = gamma
        
        # Policy parameters (simple linear policy)
        # For each state, probability of each action
        self.theta = np.random.randn(n_states, n_actions) * 0.1
    
    def policy(self, state):
        """Get action probabilities using softmax"""
        logits = self.theta[state]
        exp_logits = np.exp(logits - np.max(logits))  # Numerical stability
        probs = exp_logits / np.sum(exp_logits)
        return probs
    
    def select_action(self, state):
        """Sample action from policy"""
        probs = self.policy(state)
        return np.random.choice(self.n_actions, p=probs)
    
    def update(self, states, actions, rewards):
        """REINFORCE update"""
        T = len(states)
        returns = []
        
        # Calculate discounted returns
        G = 0
        for t in reversed(range(T)):
            G = rewards[t] + self.gamma * G
            returns.insert(0, G)
        
        # Update policy parameters
        for t in range(T):
            state = states[t]
            action = actions[t]
            G_t = returns[t]
            
            # Policy gradient
            probs = self.policy(state)
            grad_log_prob = np.zeros(self.n_actions)
            grad_log_prob[action] = 1.0 / (probs[action] + 1e-8)  # Avoid division by zero
            
            # REINFORCE update
            self.theta[state] += self.lr * G_t * grad_log_prob * probs

print("Environment and REINFORCE Agent classes defined!")


In [ ]:
# Train REINFORCE agent
env = SimpleEnv()
agent = REINFORCEAgent(n_states=10, n_actions=2, learning_rate=0.01, gamma=0.99)

num_episodes = 500
rewards_per_episode = []
episode_lengths = []

for episode in range(num_episodes):
    state = env.reset()
    states = []
    actions = []
    rewards = []
    done = False
    
    # Generate episode
    while not done:
        action = agent.select_action(state)
        next_state, reward, done = env.step(action)
        
        states.append(state)
        actions.append(action)
        rewards.append(reward)
        
        state = next_state
    
    # Update policy
    agent.update(states, actions, rewards)
    
    # Track performance
    total_reward = sum(rewards)
    rewards_per_episode.append(total_reward)
    episode_lengths.append(len(states))
    
    if (episode + 1) % 100 == 0:
        avg_reward = np.mean(rewards_per_episode[-100:])
        avg_length = np.mean(episode_lengths[-100:])
        print(f"Episode {episode+1}: Avg Reward = {avg_reward:.2f}, Avg Length = {avg_length:.2f}")

print(f"\nTraining complete!")
print(f"Average reward (last 100 episodes): {np.mean(rewards_per_episode[-100:]):.2f}")


## Learning Progress

Let's visualize the learning progress.


In [ ]:
# Plot learning curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Rewards
axes[0].plot(rewards_per_episode, alpha=0.6, linewidth=0.5)
window = 50
if len(rewards_per_episode) >= window:
    moving_avg = np.convolve(rewards_per_episode, np.ones(window)/window, mode='valid')
    axes[0].plot(range(window-1, len(rewards_per_episode)), moving_avg, 
                color='red', linewidth=2, label=f'Moving Average ({window})')
axes[0].set_xlabel('Episode')
axes[0].set_ylabel('Total Reward')
axes[0].set_title('Reward per Episode')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Episode lengths
axes[1].plot(episode_lengths, alpha=0.6, linewidth=0.5, color='green')
if len(episode_lengths) >= window:
    moving_avg_len = np.convolve(episode_lengths, np.ones(window)/window, mode='valid')
    axes[1].plot(range(window-1, len(episode_lengths)), moving_avg_len, 
                color='red', linewidth=2, label=f'Moving Average ({window})')
axes[1].set_xlabel('Episode')
axes[1].set_ylabel('Episode Length')
axes[1].set_title('Episode Length')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Policy Visualization

Let's visualize the learned policy.


In [ ]:
# Visualize learned policy
states_range = range(agent.n_states)
action_probs = np.array([agent.policy(s) for s in states_range])

plt.figure(figsize=(10, 6))
x = np.arange(len(states_range))
width = 0.35

plt.bar(x - width/2, action_probs[:, 0], width, label='Action 0 (Left)', alpha=0.7)
plt.bar(x + width/2, action_probs[:, 1], width, label='Action 1 (Right)', alpha=0.7)

plt.xlabel('State')
plt.ylabel('Probability')
plt.title('Learned Policy (Action Probabilities)')
plt.xticks(x, states_range)
plt.legend()
plt.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# Show policy for each state
print("Learned Policy:")
for state in range(min(5, agent.n_states)):  # Show first 5 states
    probs = agent.policy(state)
    print(f"  State {state}: P(Left)={probs[0]:.3f}, P(Right)={probs[1]:.3f}")


## Validation & Testing

Let's test the learned policy and compare with value-based methods.


In [ ]:
# Test learned policy
test_episodes = 10
test_rewards = []

for episode in range(test_episodes):
    env_test = SimpleEnv()
    state = env_test.reset()
    total_reward = 0
    done = False
    
    while not done:
        action = agent.select_action(state)
        next_state, reward, done = env_test.step(action)
        state = next_state
        total_reward += reward
    
    test_rewards.append(total_reward)

print("Test Results:")
print(f"  Average reward: {np.mean(test_rewards):.2f}")
print(f"  Std reward: {np.std(test_rewards):.2f}")

# Compare different learning rates
learning_rates = [0.001, 0.01, 0.1]
lr_results = []

for lr in learning_rates:
    env_lr = SimpleEnv()
    agent_lr = REINFORCEAgent(n_states=10, n_actions=2, learning_rate=lr, gamma=0.99)
    
    # Train for fewer episodes
    for episode in range(200):
        state = env_lr.reset()
        states, actions, rewards = [], [], []
        done = False
        
        while not done:
            action = agent_lr.select_action(state)
            next_state, reward, done = env_lr.step(action)
            states.append(state)
            actions.append(action)
            rewards.append(reward)
            state = next_state
        
        agent_lr.update(states, actions, rewards)
    
    # Test
    test_reward = 0
    state = env_lr.reset()
    done = False
    while not done:
        action = agent_lr.select_action(state)
        next_state, reward, done = env_lr.step(action)
        state = next_state
        test_reward += reward
    
    lr_results.append({'lr': lr, 'reward': test_reward})
    print(f"  Learning rate {lr}: Test reward = {test_reward:.2f}")

# Assertions
assert np.mean(test_rewards) > -50, "Agent should learn reasonable policy"
print("\n✓ Validation checks passed")


## Summary & Key Takeaways

### Key Concepts Learned

1. **Policy Gradient Basics**
   - Direct policy optimization
   - Uses gradient ascent on expected return
   - Policy gradient theorem provides gradient
   - REINFORCE: Monte Carlo policy gradient

2. **REINFORCE Algorithm**
   - Generate episode following current policy
   - Calculate returns (discounted sum of rewards)
   - Update policy using gradient of log-probability
   - High variance but unbiased

3. **Policy vs Value Methods**
   - **Policy-based**: Direct policy optimization
   - **Value-based**: Learn value function, derive policy
   - **Actor-Critic**: Combines both (reduces variance)

4. **Key Hyperparameters**
   - **learning_rate**: Step size for policy updates
   - **gamma**: Discount factor
   - **baseline**: Variance reduction (optional)

### When to Use Policy Gradient Methods

✅ **Good for:**
- Continuous action spaces
- High-dimensional state spaces
- Stochastic policies needed
- Complex control problems
- When value function is hard to estimate
- Robotics and continuous control

❌ **Not ideal for:**
- Simple discrete problems (Q-learning better)
- When sample efficiency is critical
- Very large action spaces
- When deterministic policy is sufficient
- Real-time applications (slow)

### Next Steps

- Try **Actor-Critic** methods (reduce variance)
- Explore **PPO (Proximal Policy Optimization)** for stability
- Use **A3C** for parallel training
- Apply to **gym continuous control** environments
- Experiment with **baseline** for variance reduction
